In [1]:
%load_ext autoreload
%autoreload 2

In [7]:
from tools.utils import *
import pandas as pd
import numpy as np
import seaborn as sns
from itertools import combinations
from collections import Counter
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.interpolate import interp1d
import scipy.spatial as sp
from matplotlib import pyplot as plt
sns.set()

In [8]:
processed_data_dir = Path('processed_data/tous')
embeddings = np.loadtxt(processed_data_dir / 'embedding.tsv')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
print(len(embeddings), len(metadata))
# 84621 84621

233880 233880


In [9]:
metadata['contract_id'] = metadata.apply(lambda x: f"{x['platform']}_{x['year']}", axis=1)
metadata['idx'] = range(len(metadata))

In [10]:
metadata['position'] = metadata['idx'] - metadata.groupby('contract_id')['idx'].transform('min')

In [11]:
metadata[metadata['contract_id']=='discord_20250901']

,platform,year,sentence,sentence_processed,contract_id,idx,position
15955,discord,20250901,Discord's Terms of Service Effective: Septembe...,[mask] [mask] [mask] [mask] [mask] [mask] : [m...,discord_20250901,15955,0
15956,discord,20250901,"Who we are HYPERLINK ""https://discord.com/term...","who we are hyperlink "" "" \l "" [mask] "" [mask] .",discord_20250901,15956,1
15957,discord,20250901,Age requirements and responsibility of parents...,age requirements and responsibility of parents...,discord_20250901,15957,2
15958,discord,20250901,"What you can expect from us HYPERLINK ""https:/...","what you can expect from us hyperlink "" "" \l ...",discord_20250901,15958,3
15959,discord,20250901,"Your Discord account HYPERLINK ""https://discor...","your [mask] account [mask] "" "" \l "" [mask] "" ...",discord_20250901,15959,4
...,...,...,...,...,...,...,...
16309,discord,20250901,We may send you electronic communications rela...,we may send you electronic communications rela...,discord_20250901,16309,354
16310,discord,20250901,"Where required, well get your consent before s...","where required , well get your consent before ...",discord_20250901,16310,355
16311,discord,20250901,For technical support or other assistance with...,for technical support or other assistance with...,discord_20250901,16311,356
16312,discord,20250901,Discord Inc. is located at 444 De Haro Street...,[mask] [mask] is located at [mask] [mask] [mas...,discord_20250901,16312,357


In [12]:
#metadata.groupby('platform').agg({'contract_id': 'nunique', 'year': 'nunique'}).sort_values('contract_id', ascending=False)

In [13]:
metadata.sort_values(['platform', 'year', 'position'], inplace=True)

In [14]:
metadata.columns

Index(['platform', 'year', 'sentence', 'sentence_processed', 'contract_id',
       'idx', 'position'],
      dtype='object')

In [15]:
from tools.plasticity_measures import *

In [ ]:
from tqdm import tqdm
platforms = metadata['platform'].unique()

results_platform = []
for platform in tqdm(platforms):
    platform_ids = list(metadata[metadata['platform']==platform]['contract_id'].unique())
    for p1, p2 in list(zip(platform_ids[:-1], platform_ids[1:])):
        X_text = list(metadata[metadata['contract_id']==p1]['sentence'].values)
        Y_text = list(metadata[metadata['contract_id']==p2]['sentence'].values)
        X = embeddings[metadata['contract_id']==p1]
        Y = embeddings[metadata['contract_id']==p2]
        #df = contract_reading_effort_density(X_text,X, Y_text, Y)
        #df = assignment_based_effort(X, Y)
        df['date'] = p2.split('_')[-1]
        results_platform.append(df)

  0%|          | 0/50 [00:00<?, ?it/s]/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
 40%|████      | 20/50 [2:33:36<3:50:24, 460.81s/it]


IndexError: index out of range in self

In [ ]:
df_all = pd.concat([pd.DataFrame.from_dict(d, orient='index').T for d in results_platform], ignore_index=True, axis=0)
df_all['year'] = df_all['date'].apply(lambda x: int(x[:4]))

In [ ]:
df_all.columns

In [ ]:
import seaborn as sns
sns.lineplot(data=df_all, x='year', y='total_effort')

In [ ]:
df_all.groupby('year').agg({'total_effort': 'sum' }).plot()